# SO-ARM101 · 카메라를 보고 공 하나를 플레이트로 옮기기

**Colab GPU · MuJoCo/MJWarp · 카메라 RGB + 관절 상태 · CNN/PPO · 브라우저 뷰어**

환경·영상·학습·뷰어 코드는 GitHub의 `environment.py`, `learning.py`, `viewer.py`에 있습니다.
노트북은 저장소 다운로드와 설정·실행을 담당합니다.

정책은 시뮬레이션 손목 카메라 영상과 로봇의 관절 상태로 팔 다섯 관절·집게의 목표 각도 변화량을 선택합니다.
정책과 가치망에는 공·플레이트의 정확한 좌표, 공 속도나 접촉·성공 판정 이력을 넣지 않습니다.
시뮬레이터의 정확한 상태는 물리 계산·보상·성공 판정에만 사용합니다.

과제는 공 하나를 집어 벽이 없는 16×16cm 플레이트에 놓고, 실제 지지된 상태를 0.5초 유지하는 것입니다.
공의 시작 위치를 바꾸어 한 위치의 동작을 외우는 데만 의존하지 않도록 구성합니다.

**이 변경본은 실행·구문·물리·GPU·학습·영상·뷰어 검증을 수행하지 않았습니다.**
T4/L4에서의 속도·메모리 사용량과 학습 성공률은 측정하지 않았습니다.

기존 숫자 상태 정책·6공 과제 정책·두 카메라 영상 정책은 호환되지 않습니다.
Colab GPU 런타임을 새로 시작하고 1번부터 실행하며 `CHECKPOINT_PATH`는 비워 두세요.


## 1. GitHub 저장소 받기

`REPO_URL`에는 코드를 올린 저장소의 HTTPS 주소를 입력합니다. 아래 셀은 공개 저장소의 인증 없는 복제를 기준으로 합니다.
비공개 저장소는 Colab에서 접근할 수 있는 인증 설정이 별도로 필요합니다.
같은 런타임에서 다시 실행하면 내려받은 폴더를 사용합니다. `GITHUB_REF`를 지정하면 해당 버전을 가져옵니다.

In [ ]:
#@title 1. 저장소 주소를 입력하고 내려받기
REPO_URL = "https://github.com/jjeong3150/so101-rl-pjt.git" #@param {type:"string"}
GITHUB_REF = "main" #@param {type:"string"}

from pathlib import Path
import importlib
import json
import subprocess
import sys

if "runner" in globals() and runner.thread is not None and runner.thread.is_alive():
    raise RuntimeError("먼저 학습을 중지한 뒤 코드를 다시 불러오세요.")
if "viewer" in globals():
    viewer.close()
if not REPO_URL.strip():
    raise ValueError("REPO_URL에 사용할 GitHub 저장소 주소를 입력하세요.")
REPO_DIR = Path("/content/so101_mjwarp_rl")
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL.strip(), str(REPO_DIR)])
else:
    print("이미 내려받은 폴더를 사용합니다:", REPO_DIR)
if GITHUB_REF.strip():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF.strip()])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"])
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
for module_name in ("environment", "learning", "viewer"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print("사용할 커밋:", REPO_COMMIT)

## 2. 패키지 설치

설치 목록은 저장소의 `requirements-colab.txt`에서 관리합니다.
다른 버전의 라이브러리를 이미 불러온 런타임이라면 새 GPU 런타임에서 시작하세요.

In [ ]:
#@title 2. 저장소의 설치 목록 사용
import importlib.util
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements-colab.txt")])
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.9.1",
                           "--index-url", "https://download.pytorch.org/whl/cu126"])
print("패키지 설치 셀이 끝났습니다.")

## 3. Python 모듈 불러오기

`environment.py`는 물리·보상·카메라 렌더링·프레임 보관,
`learning.py`는 CNN/PPO, `viewer.py`는 3D 화면과 정책 입력 영상 표시를 담당합니다.


In [ ]:
#@title 3. 환경·학습·뷰어 모듈 불러오기
importlib.invalidate_caches()
import copy
import json
import time
import numpy as np
import torch
import warp as wp
from google.colab import output
from IPython.display import display, Markdown, clear_output
from environment import SceneBundle, CPUEnv, WarpEnv, evaluate_policy, TASK_ID, NUM_BALLS, REWARD_GAMMA, OBSERVATION_SPEC
from learning import ActorCritic, PPORunner, DEFAULT_CONFIG
from viewer import LiveSO101Viewer
DEVICE = "cuda:0"
if not torch.cuda.is_available():
    raise RuntimeError("Colab의 GPU 런타임을 선택하세요.")
torch.cuda.set_device(0)
wp.init()
print("GPU:", torch.cuda.get_device_name(0))

## 4. 공식 로봇 모델 받기

원본은 MuJoCo Menagerie의 SO-ARM101 모델이며, 원본 커밋과 다운로드 파일 목록을 고정합니다.
팔의 구동력 제한은 원본 모델을 사용합니다. 공 하나와 출발 패드, 벽이 없는 평평한 플레이트를 장면에 추가합니다. 정책 영상에는 원본 손목 카메라만 사용합니다.

In [ ]:
#@title 4-1. 저장소의 원본 모델 파일 목록
ASSET_MANIFEST = json.loads((REPO_DIR / "assets_manifest.json").read_text(encoding="utf-8"))

In [ ]:
#@title 4-2. 로봇 모델 다운로드
import concurrent.futures
import hashlib
import os
import urllib.request
from pathlib import PurePosixPath

ASSET_DIR = Path("/content/so101_assets")
ASSET_DIR.mkdir(parents=True, exist_ok=True)
commit = ASSET_MANIFEST["commit"]
base_url = f"https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/{commit}/"
def download_asset(entry):
    relative = PurePosixPath(entry["path"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("모델 폴더 안의 상대 경로가 필요합니다.")
    destination = ASSET_DIR.joinpath(*relative.parts)
    expected = entry["sha256"]
    if destination.exists() and hashlib.sha256(destination.read_bytes()).hexdigest() == expected:
        return
    request = urllib.request.Request(base_url + relative.as_posix(), headers={"User-Agent": "SO101-Colab-RL/1"})
    with urllib.request.urlopen(request, timeout=90) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise RuntimeError(f"다운로드 파일이 원본과 다릅니다: {relative}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".download")
    temporary.write_bytes(payload)
    os.replace(temporary, destination)
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(download_asset, ASSET_MANIFEST["files"]))
print("원본 모델 다운로드 완료:", commit)

## 5. 환경 수와 학습 설정

정책 입력은 **카메라 RGB 영상과 관절 관련 수치 25개**입니다.
정책 영상에는 손목 `wrist_cam` 하나만 사용합니다. 고정 외부 카메라는 사용하지 않습니다.
손목 RGB는 64×64이며 최근 2프레임을 묶어 총 6채널로 입력합니다. 제어는 50Hz, 영상은 4제어 스텝마다 약 12.5Hz로 갱신합니다.

수치는 관절 각도6·속도6·누적 목표 각도6·이전 명령6·남은 시간1입니다.
공/목표 좌표·공 속도·손끝의 월드 좌표·접촉 및 집기 성공 이력은 정책과 가치망 입력에서 제외합니다.
작은 CNN이 영상 특징을 만들고 관절 수치와 합쳐 PPO의 행동·가치 예측에 사용합니다.
보상과 성공 판정만 시뮬레이터 내부의 정확한 상태를 사용합니다.

초기 손목은 이전 자세에서 180도 반전한 값을 유지합니다. 이후 손목 각도는 정책이 선택합니다.
공 시작 위치는 가로·세로 각각 최대 1.5cm 범위에서 바꿉니다.

| 보상 항목 | 값 |
|---|---:|
| 접근·잡기·들기·운반·놓고 물러나기 진척 | `3 × (0.995 × 다음 점수 − 이전 점수)` |
| 최초 실제 들어 올림 | +3, 에피소드당 한 번 |
| 플레이트에 놓기 성공 | +50, 에피소드 종료 |
| 비정상 상태·작업 공간 이탈 | −10, 에피소드 종료 |
| 시간 경과 | 제어 스텝마다 −0.002 |
| 큰 명령 / 급격한 명령 변화 | `−0.001 × 평균(a²)`, `−0.002 × 평균((a−이전a)²)` |

성공 조건은 기존과 같습니다. 양쪽 집게로 공을 잡아 4cm 이상 들어 올린 이력이 있고,
공 전체가 플레이트 가장자리에서 5mm 이상 안쪽에 놓여 실제 플레이트 지지·집게 비접촉·낮은 속도를 0.5초 유지해야 합니다.
보상은 처음부터 모두 적용합니다. 커리큘럼·시연 데이터·미리 정한 집기 경로는 없습니다.

영상 생성과 CNN 비용을 줄이기 위한 초기 설정은 **T4 환경4개 / L4 환경8개, rollout64스텝**입니다.
실제 처리 속도나 메모리는 측정하지 않았습니다. 메모리가 부족하면 환경을 2개로 줄일 수 있습니다.
`CHECKPOINT_PATH`에는 이번 영상 과제에서 저장한 정책만 지정할 수 있습니다.


In [ ]:
#@title 5. 환경 수와 학습량 선택
ENV_PRESET = "자동" #@param ["자동", "2개", "4개", "8개", "16개"]
UPDATES = 1000 #@param {type:"integer"}
MAX_EPISODE_STEPS = 1000 #@param {type:"integer"}
EVAL_EVERY = 100 #@param {type:"integer"}
SEED = 7 #@param {type:"integer"}
CHECKPOINT_PATH = "" #@param {type:"string"}

if "runner" in globals() and runner.thread is not None and runner.thread.is_alive():
    raise RuntimeError("진행 중인 학습을 먼저 중지하세요.")
if "viewer" in globals():
    viewer.close()
gpu_name = torch.cuda.get_device_name(0).upper()
num_envs = 8 if "L4" in gpu_name else 4
if ENV_PRESET != "자동":
    num_envs = int(ENV_PRESET.removesuffix("개"))
config = dict(DEFAULT_CONFIG, num_envs=num_envs, horizon=64, minibatch_size=128, updates=UPDATES,
              eval_every=EVAL_EVERY, eval_episodes=1, seed=SEED)
bundle = SceneBundle(ASSET_DIR, max_episode_steps=MAX_EPISODE_STEPS, visual_device=DEVICE)
RUN_DIR = Path("/content/so101_one_ball_vision_runs") / time.strftime("%Y%m%d_%H%M%S", time.gmtime())
def make_training_env():
    return WarpEnv(bundle, num_envs=config["num_envs"], device=DEVICE, seed=config["seed"], jitter=.015, capture_graph=False)
def fixed_evaluator(policy, mode="learned", seed=2026):
    return evaluate_policy(bundle, policy, mode=mode, episodes=config["eval_episodes"], seed=seed, jitter=.015)
runner = PPORunner(config, make_env=make_training_env, evaluator=fixed_evaluator,
                   output_dir=RUN_DIR, device=DEVICE, observation_dim=bundle.observation_dim,
                   action_dim=bundle.action_dim,
                   metadata={"task": TASK_ID,
                             "asset_manifest": ASSET_MANIFEST, "source_commit": REPO_COMMIT, "max_episode_steps": MAX_EPISODE_STEPS,
                             "action": "policy_selected_joint_target_increments", "cuda_graph_capture": False,
                             "observation_spec": OBSERVATION_SPEC, "policy_input": "camera_rgb_and_joint_proprio"})
if CHECKPOINT_PATH.strip():
    runner.load_checkpoint(CHECKPOINT_PATH.strip(), restore_optimizer=True)
    print("이전 정책을 불러왔습니다. 업데이트 횟수는 이번에 추가할 학습량입니다.")
print("과제:", TASK_ID)
print("환경:", num_envs, "| 관절 수치:", bundle.observation_dim, "| 행동:", bundle.action_dim)
print("영상 입력:", json.dumps(OBSERVATION_SPEC, ensure_ascii=False))
print("이번 실행 전이 예산:", num_envs * config["horizon"] * config["updates"])
print("저장 위치:", RUN_DIR)

## 6. 웹 3D 뷰어와 정책 입력 영상

3D 화면은 사람이 관찰하는 뷰어입니다. 오른쪽 **정책이 보는 영상**에는 정책에 실제로 입력한 최신 RGB 프레임을 표시합니다.
화면의 목표 사각형·성공 표시 같은 오버레이는 정책의 카메라 영상에 포함되지 않습니다.

뷰어 물리는 별도 CPU 환경에서 계산하고, 카메라 영상은 학습과 같은 MJWarp 렌더러로 GPU에서 만듭니다.
카메라 회전·확대, 일시정지, 전체 초기화를 사용할 수 있습니다.
학습 시작 시 정책 / 최신 정책 / 무작위 행동을 비교하며, 정책은 에피소드 시작 때 불러옵니다.
뷰어 제어 스텝은 시뮬레이션 진행량이며 PPO 업데이트 횟수와 다릅니다.

손목 카메라 렌더러를 준비하기 전에 3D 모델의 초기 관절 자세부터 표시합니다.
준비 중에는 초기 자세가 유지되며, 정책 평가가 시작되면 제어 스텝이 진행됩니다.


In [ ]:
#@title 6-1. Colab 내부 뷰어
if "viewer" in globals():
    viewer.close()
viewer = LiveSO101Viewer(runner, bundle, port=8081, seed=2026, jitter=.015)
output.serve_kernel_port_as_iframe(viewer.server.get_port(), path="/", width="100%",
                                  height="700", cache_in_notebook=False)

In [ ]:
#@title 6-2. 본인 세션의 별도 브라우저 탭 링크
SHOW_OWN_TAB_LINK = True #@param {type:"boolean"}
if SHOW_OWN_TAB_LINK:
    try:
        import html
        from IPython.display import HTML
        private_viewer_url = output.eval_js(f"google.colab.kernel.proxyPort({viewer.server.get_port()})")
        if private_viewer_url:
            safe_url = html.escape(str(private_viewer_url), quote=True)
            display(HTML(f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer">내 3D 뷰어를 별도 탭에서 열기</a>'))
        else:
            print("별도 탭 주소를 가져오지 못했습니다. 내부 뷰어를 사용하세요.")
    except Exception:
        print("별도 탭 주소를 가져오지 못했습니다. 내부 뷰어를 사용하세요.")

별도 탭은 현재 로그인한 본인 Colab 세션의 프록시 주소를 사용합니다. 런타임을 열어 두세요.
브라우저 보안 설정에 따라 별도 탭 연결이 안 될 수 있으며, 그때는 내부 뷰어를 사용하세요.
외부에 공개하는 터널은 만들지 않습니다.

## 7. GPU 병렬 강화학습 시작

GPU 커널 준비 이후 경험 수집과 PPO 업데이트가 진행됩니다. 처음 평가가 끝나기까지 기다릴 수 있습니다.
학습과 웹 뷰어는 각각 별도의 스레드에서 진행됩니다.

호환성을 위해 CUDA 그래프 캡처 최적화는 끕니다. 물리 계산과 카메라 렌더링은 GPU의 MJWarp에서 수행합니다.
RGB 영상 생성에는 MuJoCo Warp 3.16.0의 렌더 API를 사용하며 OpenGL/EGL은 필요하지 않습니다.

In [ ]:
#@title 7. 학습 시작
runner.start()
print("학습을 시작했습니다. 아래 표시 셀을 중단해도 학습 스레드는 계속 실행됩니다.")

## 8. 학습 진행 보기

보상과 함께 실제 들어 올림 비율, 놓기 이력 비율, 최종 성공률을 확인합니다.
공을 들어 올린 기록만으로 플레이트 옮기기에 성공한 것은 아닙니다.
기본 평가는 고정된 한 조건의 결과이며 다양한 초기 조건에 대한 성공률 추정은 아닙니다.
`DISPLAY_LIVE=False`면 현재 상태만 표시합니다. 표시 중단은 학습 중단과 다릅니다. 학습 중단은 9번입니다.


In [ ]:
#@title 8. 보상·들어 올림·성공률 보기
DISPLAY_LIVE = True #@param {type:"boolean"}
import matplotlib.pyplot as plt
def show_evaluation():
    baseline = runner.baseline or {}
    latest = runner.final_evaluation or {}
    rows = [("학습 시작 시 정책", baseline.get("learned")),
            ("최신 정책", latest.get("learned")),
            ("무작위 행동", latest.get("random"))]
    lines = ["| 평가 방식 | 성공 | 들어 올림 | 놓기 이력 | 평균 보상 |", "|---|---:|---:|---:|---:|"]
    for label, result in rows:
        if result:
            lines.append(f"| {label} | {result.get('success_rate', 0):.0%} | {result.get('lift_rate', 0):.0%} | {result.get('release_rate', 0):.0%} | {result.get('mean_return', 0):.1f} |")
        else:
            lines.append(f"| {label} | 대기 | — | — | — |")
    display(Markdown("\n".join(lines)))
try:
    while True:
        with runner.lock:
            history = copy.deepcopy(runner.history)
        clear_output(wait=True)
        print("상태:", runner.status, "| 업데이트:", runner.completed_updates, "| 누적 전이:", runner.total_steps)
        print("브라우저 접속:", len(viewer.server.get_clients()))
        show_evaluation()
        if history:
            x = [row["steps"] for row in history]
            fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
            axes[0].plot(x, [row.get("train_mean_return") for row in history])
            axes[0].set(xlabel="Transitions", ylabel="Episode return")
            axes[1].plot(x, [row.get("train_success_rate") for row in history], label="Success")
            axes[1].plot(x, [row.get("train_lift_rate") for row in history], label="Lift")
            axes[1].plot(x, [row.get("train_release_rate") for row in history], label="Release")
            axes[1].set(xlabel="Transitions", ylabel="Episode rates", ylim=(-.05, 1.05))
            axes[1].legend()
            axes[2].plot(x, [row.get("ppo_loss") for row in history])
            axes[2].set(xlabel="Transitions", ylabel="PPO loss")
            fig.tight_layout()
            display(fig)
            plt.close(fig)
        if runner.error:
            print(runner.error)
            break
        if viewer.error:
            print("뷰어 오류:", viewer.error)
        if not DISPLAY_LIVE or runner.thread is None or not runner.thread.is_alive():
            break
        time.sleep(3)
except KeyboardInterrupt:
    print("표시만 중단했습니다. 학습을 멈추려면 9번을 실행하세요.")


## 9. 중지와 정책 저장

체크하고 실행하면 중지를 요청합니다. 저장까지 끝난 뒤 결과를 다운로드하세요.
체크포인트는 정책과 최적화 정보를 보관합니다. 이어 학습할 때 물리 상태와 난수 흐름은 새로 시작합니다.

In [ ]:
#@title 9. 학습 중지 요청
STOP_TRAINING = False #@param {type:"boolean"}
if STOP_TRAINING:
    runner.stop_event.set()
    if runner.thread is not None:
        runner.thread.join(timeout=10)
    print("상태:", runner.status)
else:
    print("학습을 계속합니다. 중지하려면 체크하고 이 셀을 실행하세요.")
print("결과 폴더:", RUN_DIR)

## 10. 결과 다운로드

학습 완료나 중지 후 체크하여 실행합니다. 저장한 `latest.pt`를 다음 실행의 `CHECKPOINT_PATH`에 지정하면
이번 단일 공 과제의 정책과 최적화 상태를 불러와 추가 학습합니다. 기존 숫자 상태·6공 과제·두 카메라 영상 정책은 호환되지 않습니다. 물리 에피소드와 난수 흐름은 새로 시작합니다.

In [ ]:
#@title 10. 정책과 실행 코드 ZIP 다운로드
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    if runner.thread is None or runner.thread.is_alive():
        raise RuntimeError("학습 완료 또는 중지 후 다운로드하세요.")
    runner.save("latest.pt")
    included = RUN_DIR / "included_code"
    included.mkdir(parents=True, exist_ok=True)
    for name in ("environment.py", "learning.py", "viewer.py", "assets_manifest.json", "requirements-colab.txt"):
        shutil.copy2(REPO_DIR / name, included / name)
    (RUN_DIR / "source_commit.txt").write_text(REPO_COMMIT, encoding="utf-8")
    archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    files.download(archive)

In [ ]:
#@title 11. 뷰어 종료 — 선택
CLOSE_VIEWER = False #@param {type:"boolean"}
if CLOSE_VIEWER:
    viewer.close()
    print("뷰어를 닫았습니다. GPU 사용을 마치면 Colab 런타임도 연결 해제하세요.")

원본과 참고: [MuJoCo Menagerie SO-ARM101](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotstudio_so101),
[MuJoCo Warp](https://github.com/google-deepmind/mujoco_warp), [NVIDIA Warp](https://github.com/NVIDIA/warp),
[mjviser](https://github.com/mujocolab/mjviser), [Viser](https://github.com/viser-project/viser),
[PPO 논문](https://arxiv.org/abs/1707.06347),
[고정 버전 MJWarp 카메라 렌더링 튜토리얼](https://github.com/google-deepmind/mujoco_warp/blob/v3.16.0/notebooks/tutorial.ipynb),
[진척 보상의 설계 참고](https://people.eecs.berkeley.edu/~pabbeel/cs287-fa09/readings/NgHaradaRussell-shaping-ICML1999.pdf).
원본 로봇 라이선스는 다운로드한 `robotstudio_so101/LICENSE`에 있습니다.

이 노트북은 시뮬레이션 학습용이며 실제 로봇을 연결하거나 움직이지 않습니다.